# Calculus and Optimization — Exercises

Companion to the note [Calculus and Optimization](Calculus%20and%20Optimization.md).

Practise gradients and Hessians, convexity, classifying stationary points, Lagrange multipliers and KKT, Taylor approximations, and the methods in the note's table (gradient descent, Newton, L-BFGS, coordinate descent): by hand and from scratch in NumPy, checked against SciPy and scikit-learn.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.optimize import minimize
from sklearn.linear_model import Lasso, LogisticRegression
rng = np.random.default_rng(0)

def rosen(x):
    return (1 - x[0]) ** 2 + 100 * (x[1] - x[0] ** 2) ** 2

## Part A · Concepts

### Exercise 1 · Which functions are convex?
*🧠 Concept · ★☆☆*

Decide which are convex on their domain and justify briefly (second derivative, or "max/sum of convex functions"):

(a) $x^2$  (b) $|x|$  (c) hinge $\max(0, 1-x)$  (d) $x^3$ on $\mathbb R$  (e) $-\log x$ on $x>0$  (f) $\log(1+e^{-x})$  (g) $\sin x$  (h) $\lVert Ax-b\rVert_2^2$

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A twice-differentiable $f$ is convex iff $f''\ge 0$ (or $H\succeq0$). The pointwise max of convex functions and non-negative sums of convex functions are convex.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Convex: (a) $f''=2$; (b) max of $x$ and $-x$; (c) max of two affine functions (SVM loss); (e) $f''=1/x^2>0$;
(f) logistic loss, $f'' = \sigma(x)(1-\sigma(x))>0$; (h) Hessian $2A^\top A\succeq0$.
Not convex: (d) $f''=6x<0$ for $x<0$; (g) $f''=-\sin x$ changes sign.
That is why linear/logistic regression and SVMs have unique global optima (up to flat directions), unlike neural networks.

</details>

### Exercise 2 · Saddle points in high dimension
*🧠 Concept · ★★☆*

At a stationary point the Hessian has $d$ eigenvalues. Suppose (as a crude model) each eigenvalue is independently positive or
negative with probability $1/2$. What is the probability that a random stationary point is a local minimum when $d=2$? When
$d=1000$? What does this suggest about the loss landscapes of deep networks, and why can plain gradient descent still escape
saddles in practice?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A local minimum needs *all* eigenvalues positive.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(\text{min}) = 2^{-d}$: $1/4$ for $d=2$, about $10^{-301}$ for $d=1000$. In high dimension almost every critical point has
some negative curvature direction, so saddles vastly outnumber local minima (and empirically the minima that exist tend to have
similar loss). At a strict saddle the gradient is zero, but any small perturbation (SGD noise, random init) has a component
along a negative-curvature direction that grows exponentially under GD, so the iterate escapes; it can however slow down
training on plateaus.

</details>

### Exercise 3 · Choosing an optimiser
*🧠 Concept · ★★☆*

For each scenario pick a method from the note's table (GD/SGD, Newton, L-BFGS, coordinate descent, EM) and justify it:

1. Logistic regression with 50 features and 10 000 samples.
2. A neural network with 10 million parameters.
3. Lasso regression with 5 000 features.
4. Fitting a Gaussian mixture model.
5. A smooth convex problem with 20 parameters where you need 12 correct digits.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Newton costs $O(d^3)$ per step and $O(d^2)$ memory; L-BFGS stores only a few vectors; coordinate descent shines when each 1-D subproblem has a closed form.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **L-BFGS** (sklearn's default `lbfgs` solver): smooth, convex, medium size; converges in tens of iterations.
2. **Mini-batch SGD/Adam**: Hessians are impossible at this size and full-batch gradients are too expensive ([[Optimizers]]).
3. **Coordinate descent**: the L1 term is non-smooth, but each 1-D problem is solved exactly by soft-thresholding (sklearn's `Lasso`).
4. **EM**: alternating closed-form E and M steps increase the likelihood monotonically ([[Gaussian Mixture Models and EM]]).
5. **Newton**: $d=20$ makes the Hessian cheap, and quadratic convergence doubles the number of correct digits per step near the optimum.

</details>

### Exercise 4 · Complementary slackness, intuitively
*🧠 Concept · ★★☆*

State the four KKT conditions for $\min f(x)$ s.t. $g_i(x)\le 0$ with Lagrangian $\mathcal L = f + \sum_i\lambda_i g_i$.
Explain complementary slackness $\lambda_i g_i(x^*)=0$ in words, and connect it to **support vectors** in the SVM dual
([[Support Vector Machines]]).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Either a constraint is active ($g_i=0$) or its multiplier is zero.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Stationarity $\nabla f(x^*) + \sum_i\lambda_i\nabla g_i(x^*) = 0$; 2. primal feasibility $g_i(x^*)\le0$;
3. dual feasibility $\lambda_i\ge0$; 4. complementary slackness $\lambda_i g_i(x^*) = 0$.

Complementary slackness: an inactive constraint (strictly satisfied) cannot "push", so its multiplier is $0$; only constraints
the solution leans against get $\lambda_i>0$. In the SVM each training point gives a margin constraint
$1 - y_i(w^\top x_i+b)\le0$. Points beyond the margin have $\alpha_i=0$; only points on or inside the margin have
$\alpha_i>0$. These are the support vectors, and $w=\sum_i\alpha_iy_ix_i$ depends on them alone.

</details>

## Part B · By hand

### Exercise 5 · Gradient and Hessian
*✍️ By hand · ★☆☆*

For $f(x,y) = x^2 + 3xy + 2y^2$ compute $\nabla f(1,1)$ and the Hessian $H$. Is $f$ convex? What kind of stationary point is the origin?

In [ ]:
grad_f11 = None   # [df/dx, df/dy] at (1,1)
H_f = None        # 2x2 nested list

_f = lambda v: v[0] ** 2 + 3 * v[0] * v[1] + 2 * v[1] ** 2
_e = 1e-5
check('gradient', grad_f11, [(_f([1 + _e, 1]) - _f([1 - _e, 1])) / (2 * _e), (_f([1, 1 + _e]) - _f([1, 1 - _e])) / (2 * _e)], tol=1e-4)
check('Hessian', H_f, [[2, 3], [3, 4]])

<details>
<summary><b>💡 Hint</b></summary>

$\partial_x f = 2x+3y$, $\partial_y f = 3x+4y$. Look at $\det H$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla f = (2x+3y,\ 3x+4y)$, so $\nabla f(1,1) = (5, 7)$. $H = \begin{pmatrix}2&3\\3&4\end{pmatrix}$, $\det H = 8-9=-1<0$:
one positive and one negative eigenvalue, so $H$ is indefinite, $f$ is **not** convex, and the origin is a **saddle**.

```python
grad_f11 = [5, 7]
H_f = [[2, 3], [3, 4]]
```

</details>

### Exercise 6 · Classifying stationary points
*✍️ By hand · ★★☆*

Find all stationary points of $f(x,y) = x^3 - 3x + y^2$ and classify each with the Hessian.
Store them as a dict `{(x, y): 'min' | 'max' | 'saddle'}`.

In [ ]:
stationary = None

def _classify(p):
    ev = np.linalg.eigvalsh(np.array([[6 * p[0], 0], [0, 2.]]))
    return 'min' if ev.min() > 0 else ('max' if ev.max() < 0 else 'saddle')
_grad = lambda p: np.array([3 * p[0] ** 2 - 3, 2 * p[1]])
if stationary is None:
    check('stationary points', None, True)
else:
    check('all points are stationary', all(np.allclose(_grad(p), 0) for p in stationary), True)
    check('found both points', len(stationary), 2)
    check('classification', all(_classify(p) == c for p, c in stationary.items()), True)

<details>
<summary><b>💡 Hint</b></summary>

$\nabla f = (3x^2-3,\ 2y)$, $H = \mathrm{diag}(6x, 2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$3x^2-3=0\Rightarrow x=\pm1$; $2y=0\Rightarrow y=0$. At $(1,0)$: $H=\mathrm{diag}(6,2)\succ0$ → local **min** ($f=-2$).
At $(-1,0)$: $H=\mathrm{diag}(-6,2)$ indefinite → **saddle**. There is no global minimum since $f\to-\infty$ as $x\to-\infty$:
a local min need not be global for non-convex $f$.

```python
stationary = {(1, 0): 'min', (-1, 0): 'saddle'}
```

</details>

### Exercise 7 · Lagrange multipliers
*✍️ By hand · ★★☆*

Minimise $f(x,y)=x^2+y^2$ subject to $x+2y=5$ using $\mathcal L = f + \lambda(x+2y-5)$. Store $x^*, y^*$ and $\lambda^*$.
Interpret the result geometrically.

In [ ]:
x_star = None; y_star = None; lam_star = None

_r = minimize(lambda v: v @ v, [0., 0.], constraints={'type': 'eq', 'fun': lambda v: v[0] + 2 * v[1] - 5})
check('x*', x_star, _r.x[0], tol=1e-4); check('y*', y_star, _r.x[1], tol=1e-4)
check('lambda*', lam_star, -2 * _r.x[0], tol=1e-4)   # from stationarity 2x + lambda = 0

<details>
<summary><b>💡 Hint</b></summary>

$\partial_x\mathcal L = 2x+\lambda=0$, $\partial_y\mathcal L = 2y+2\lambda=0$, so $x=-\lambda/2$, $y=-\lambda$. Plug into the constraint.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$x=-\lambda/2$, $y=-\lambda$; constraint: $-\lambda/2 - 2\lambda = 5 \Rightarrow \lambda^* = -2$, so $(x^*,y^*)=(1,2)$, $f^*=5$.
Geometrically, $(1,2)$ is the point of the line closest to the origin: $\nabla f=(2,4)$ is parallel to the constraint normal $(1,2)$.

```python
x_star = 1.0; y_star = 2.0; lam_star = -2.0
```

</details>

### Exercise 8 · KKT with an inequality
*✍️ By hand · ★★☆*

Solve $\min_x (x-2)^2$ s.t. $x\le 1$ with the KKT conditions ($\mathcal L = (x-2)^2 + \mu(x-1)$). Store $x^*$ and $\mu^*$.
Then redo it with the constraint $x\le 3$ and store $x^*_3$, $\mu^*_3$.

In [ ]:
x_kkt = None; mu_kkt = None
x_kkt3 = None; mu_kkt3 = None

for _c, _x, _m in [(1, x_kkt, mu_kkt), (3, x_kkt3, mu_kkt3)]:
    _r = minimize(lambda v: (v[0] - 2) ** 2, [0.], constraints={'type': 'ineq', 'fun': lambda v, c=_c: c - v[0]})
    check(f'x* (x <= {_c})', _x, _r.x[0], tol=1e-4)
    check(f'mu* (x <= {_c})', _m, -2 * (_r.x[0] - 2), tol=1e-4)   # stationarity: 2(x-2) + mu = 0

<details>
<summary><b>💡 Hint</b></summary>

Try both cases: constraint inactive ($\mu=0$, unconstrained optimum) or active ($x=1$). Keep the case satisfying all KKT conditions.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Case $\mu=0$: $x=2$, infeasible for $x\le1$. Case active: $x=1$, stationarity $2(1-2)+\mu=0\Rightarrow\mu^*=2\ge0$ ✔. So $x^*=1,\ \mu^*=2$.
With $x\le3$ the unconstrained optimum $x=2$ is feasible: constraint inactive, $x^*_3=2$, $\mu^*_3=0$ (complementary slackness).
$\mu^*=2$ is also the sensitivity: relaxing the bound to $x\le 1+\delta$ lowers the optimum by about $2\delta$.

```python
x_kkt = 1.0; mu_kkt = 2.0
x_kkt3 = 2.0; mu_kkt3 = 0.0
```

</details>

### Exercise 9 · Newton solves a quadratic in one step
*✍️ By hand · ★★☆*

For $f(x) = \tfrac12 x^\top Ax - b^\top x$ with $A\succ0$, show that one Newton step from **any** $x_0$ lands on the minimiser.
Compute it for $A=\begin{pmatrix}3&1\\1&2\end{pmatrix}$, $b=(1,1)^\top$.

In [ ]:
x_newton = None

check('minimiser', x_newton, np.linalg.solve(np.array([[3., 1.], [1., 2.]]), np.ones(2)))

<details>
<summary><b>💡 Hint</b></summary>

$\nabla f = Ax-b$, $H=A$. Write out $x_1 = x_0 - H^{-1}\nabla f(x_0)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$x_1 = x_0 - A^{-1}(Ax_0-b) = A^{-1}b$, which satisfies $\nabla f=0$; independent of $x_0$. Newton fits a local quadratic model and
jumps to its minimum, and here the model is exact. $A^{-1} = \frac15\begin{pmatrix}2&-1\\-1&3\end{pmatrix}$, so $x^* = \frac15(1, 2) = (0.2, 0.4)$.

```python
x_newton = [0.2, 0.4]
```

</details>

### Exercise 10 · Second-order Taylor approximation
*✍️ By hand · ★★☆*

Write the second-order Taylor expansion $f(x_0+\delta)\approx f(x_0) + \nabla f^\top\delta + \tfrac12\delta^\top H\delta$ of
$f(x,y) = e^{x+2y}$ around $(0,0)$ and evaluate it at $\delta=(0.1, 0.1)$. Compare with the true value.

In [ ]:
taylor2 = None

_d = np.array([0.1, 0.1]); _g = np.array([1., 2.]); _H = np.outer(_g, _g)
check('2nd-order Taylor', taylor2, 1 + _g @ _d + 0.5 * _d @ _H @ _d)

<details>
<summary><b>💡 Hint</b></summary>

$\nabla f = f\cdot(1,2)$ and $H = f\cdot\begin{pmatrix}1&2\\2&4\end{pmatrix}$; at the origin $f=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla f^\top\delta = 0.1+0.2 = 0.3$, $\delta^\top H\delta = (0.1+0.2)^2 = 0.09$, so $f\approx 1+0.3+0.045 = 1.345$, while
$e^{0.3}\approx1.34986$. The error ($\approx 0.0049$) is third order, $\approx 0.3^3/6=0.0045$. Newton's method minimises exactly this
quadratic model at every step.

```python
taylor2 = 1.345
```

</details>

### Exercise 11 · Convexity of logistic regression
*✍️ By hand · ★★★*

For $L(w) = -\sum_i\left[y_i\log\sigma(w^\top x_i) + (1-y_i)\log(1-\sigma(w^\top x_i))\right]$ show that
$\nabla L = X^\top(\sigma(Xw)-y)$ and $H = X^\top S X$ with $S=\mathrm{diag}(\sigma_i(1-\sigma_i))$. Conclude that $L$ is convex.
When is it *strictly* convex, and what happens to the MLE on linearly separable data?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Use $\sigma'(z)=\sigma(z)(1-\sigma(z))$ and that $\frac{d}{dz}[-y\log\sigma(z)-(1-y)\log(1-\sigma(z))] = \sigma(z)-y$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Per sample, $\partial\ell_i/\partial z_i = \sigma_i - y_i$ and $z_i=w^\top x_i$, so $\nabla L=\sum_i(\sigma_i-y_i)x_i = X^\top(\sigma-y)$.
Differentiating again: $H=\sum_i\sigma_i(1-\sigma_i)x_ix_i^\top = X^\top SX$. For any $v$: $v^\top Hv = \sum_i s_i(x_i^\top v)^2\ge0$,
so $H\succeq0$ and $L$ is convex. It is strictly convex if $X$ has full column rank (all $s_i>0$).
On separable data the loss can be pushed towards $0$ by scaling $w\to\infty$: no finite minimiser exists, which is why
regularisation (sklearn's default $C=1$) is needed. Newton on this problem is IRLS.

</details>

## Part C · Code from scratch

### Exercise 12 · Finite-difference gradient and Hessian
*💻 Code · ★☆☆*

Implement `num_grad(f, x, h=1e-5)` (central differences) and `num_hess(f, x, h=1e-4)`. Test them on the Rosenbrock function
`rosen` (defined in the setup) at $(-1.2, 1)$ against the analytic gradient and Hessian, which you also derive and implement as
`rosen_grad`, `rosen_hess`.

In [ ]:
def num_grad(f, x, h=1e-5):
    return None
def num_hess(f, x, h=1e-4):
    return None
def rosen_grad(x):
    return None
def rosen_hess(x):
    return None

x0_r = np.array([-1.2, 1.0])

from scipy.optimize import rosen_der, rosen_hess as _rh
check('analytic grad', rosen_grad(x0_r), rosen_der(x0_r))
check('analytic Hessian', rosen_hess(x0_r), _rh(x0_r))
check('numerical grad', num_grad(rosen, x0_r), rosen_der(x0_r), tol=1e-4)
check('numerical Hessian', num_hess(rosen, x0_r), _rh(x0_r), tol=1e-2)

<details>
<summary><b>💡 Hint 1</b></summary>

$\partial_i f\approx\frac{f(x+he_i)-f(x-he_i)}{2h}$. For the Hessian, apply central differences to `num_grad`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\nabla f = (-2(1-x)-400x(y-x^2),\ 200(y-x^2))$, $H = \begin{pmatrix}2-400y+1200x^2 & -400x\\ -400x & 200\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Central differences have $O(h^2)$ error versus $O(h)$ for forward differences; too small an $h$ amplifies round-off. This is how you gradient-check a hand-written backward pass ([[Gradient Descent]]).

```python
def num_grad(f, x, h=1e-5):
    x = np.asarray(x, float); g = np.zeros_like(x)
    for i in range(len(x)):
        e = np.zeros_like(x); e[i] = h
        g[i] = (f(x + e) - f(x - e)) / (2 * h)
    return g
def num_hess(f, x, h=1e-4):
    x = np.asarray(x, float); n = len(x); H = np.zeros((n, n))
    for i in range(n):
        e = np.zeros(n); e[i] = h
        H[:, i] = (num_grad(f, x + e) - num_grad(f, x - e)) / (2 * h)
    return (H + H.T) / 2
def rosen_grad(x):
    return np.array([-2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0] ** 2), 200 * (x[1] - x[0] ** 2)])
def rosen_hess(x):
    return np.array([[2 - 400 * x[1] + 1200 * x[0] ** 2, -400 * x[0]], [-400 * x[0], 200.]])

x0_r = np.array([-1.2, 1.0])
```

</details>

### Exercise 13 · Newton's method versus gradient descent on Rosenbrock
*💻 Code · ★★☆*

Starting from $(-1.2, 1)$, run (a) pure Newton steps until $\lVert\nabla f\rVert<10^{-8}$ and (b) gradient descent with
$\eta=10^{-3}$ for up to $50\,000$ steps with the same stopping rule. Store the final Newton iterate in `x_newton_r`, its number
of iterations in `it_newton`, and the GD iteration count in `it_gd`.

In [ ]:
x_newton_r = None; it_newton = None; it_gd = None

check('Newton reaches (1, 1)', x_newton_r, minimize(rosen, x0_r, method='BFGS', options={'gtol': 1e-10}).x, tol=1e-5)
check('Newton needs < 20 iterations, GD > 1000', None if it_newton is None else (it_newton < 20 and it_gd > 1000), True)

<details>
<summary><b>💡 Hint</b></summary>

Newton: `x -= np.linalg.solve(rosen_hess(x), rosen_grad(x))`. GD with a larger $\eta$ diverges because the curvature is up to about $10^3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Newton converges in about 6 iterations (quadratic convergence near $(1,1)$); GD needs tens of thousands of steps along the curved, ill-conditioned valley (or does not reach $10^{-8}$ at all within the budget). The price of Newton is forming and solving with the $d\times d$ Hessian.

```python
x = x0_r.copy(); it_newton = 0
while np.linalg.norm(rosen_grad(x)) > 1e-8 and it_newton < 100:
    x = x - np.linalg.solve(rosen_hess(x), rosen_grad(x)); it_newton += 1
x_newton_r = x
x = x0_r.copy(); it_gd = 0
while np.linalg.norm(rosen_grad(x)) > 1e-8 and it_gd < 50_000:
    x = x - 1e-3 * rosen_grad(x); it_gd += 1
print('Newton:', it_newton, 'GD:', it_gd, 'GD final:', x)
```

</details>

### Exercise 14 · Condition number and gradient descent
*💻 Code · ★★☆*

For $f(x)=\tfrac12x^\top\mathrm{diag}(1,\kappa)\,x$ and $\kappa\in\{1, 10, 100, 1000\}$, run GD from $x_0=(1,1)$ with the best fixed
step $\eta = 2/(\lambda_{\min}+\lambda_{\max})$ until $\lVert x\rVert < 10^{-6}$. Store the iteration counts in `iters_kappa`
(list). Compare with the theoretical rate $\left(\frac{\kappa-1}{\kappa+1}\right)^t$.

In [ ]:
iters_kappa = None

_pred = [1 if k == 1 else int(np.ceil(np.log(1e-6 / np.sqrt(2)) / np.log((k - 1) / (k + 1)))) for k in [1, 10, 100, 1000]]
check('iterations match theory (within 2)', None if iters_kappa is None else np.max(np.abs(np.array(iters_kappa) - _pred)) <= 2, True)

<details>
<summary><b>💡 Hint</b></summary>

Each coordinate evolves as $x_i\leftarrow(1-\eta\lambda_i)x_i$; with the optimal $\eta$ both factors have modulus $\frac{\kappa-1}{\kappa+1}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Counts grow roughly linearly in $\kappa$ (1, about 70, about 700, about 7000): GD's rate depends on the condition number. Newton (or preconditioning / feature standardisation) removes this dependence, which is why scaling features matters.

```python
iters_kappa = []
for k in [1, 10, 100, 1000]:
    lam = np.array([1., k]); eta = 2 / (1 + k)
    x = np.ones(2); t = 0
    while np.linalg.norm(x) >= 1e-6:
        x = x - eta * lam * x; t += 1
    iters_kappa.append(t)
print(iters_kappa)
```

</details>

### Exercise 15 · Coordinate descent for the Lasso
*💻 Code · ★★★*

Implement `lasso_cd(X, y, alpha, n_iter=200)` for sklearn's objective
$\frac{1}{2N}\lVert y-Xw\rVert^2 + \alpha\lVert w\rVert_1$ (no intercept) by cycling over coordinates with the soft-threshold update
$$w_j \leftarrow \frac{S(\rho_j,\ N\alpha)}{\lVert x_j\rVert^2},\quad \rho_j = x_j^\top(y - Xw + x_jw_j),\quad S(z,t) = \mathrm{sign}(z)\max(|z|-t, 0).$$
Compare with `Lasso(alpha, fit_intercept=False)`. How many coefficients are exactly zero?

In [ ]:
def lasso_cd(X, y, alpha, n_iter=200):
    # TODO
    return None

X_l = rng.normal(size=(100, 10))
w_true = np.array([3., -2., 0, 0, 1.5, 0, 0, 0, 0, 0])
y_l = X_l @ w_true + 0.5 * rng.normal(size=100)
w_cd = lasso_cd(X_l, y_l, alpha=0.1)

check('lasso CD', w_cd, Lasso(alpha=0.1, fit_intercept=False, tol=1e-10, max_iter=100000).fit(X_l, y_l).coef_, tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

Keep the residual $r = y - Xw$ up to date: before updating $w_j$ add back $x_jw_j$, after updating subtract $x_jw_j^{\text{new}}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each 1-D subproblem $\min_{w_j}\frac{1}{2N}\lVert r_{-j} - x_jw_j\rVert^2+\alpha|w_j|$ is solved exactly by soft-thresholding; the threshold sets small coefficients **exactly** to zero (sparsity), which plain gradient descent on the non-smooth L1 term does not. Here all 7 irrelevant coefficients end up exactly 0, and the 3 true ones are shrunk slightly towards 0.

```python
def lasso_cd(X, y, alpha, n_iter=200):
    N, d = X.shape
    w = np.zeros(d); r = y.astype(float).copy()
    col_sq = (X ** 2).sum(0)
    for _ in range(n_iter):
        for j in range(d):
            r += X[:, j] * w[j]
            rho = X[:, j] @ r
            w[j] = np.sign(rho) * max(abs(rho) - N * alpha, 0) / col_sq[j]
            r -= X[:, j] * w[j]
    return w

X_l = rng.normal(size=(100, 10))
w_true = np.array([3., -2., 0, 0, 1.5, 0, 0, 0, 0, 0])
y_l = X_l @ w_true + 0.5 * rng.normal(size=100)
w_cd = lasso_cd(X_l, y_l, alpha=0.1)
print(np.round(w_cd, 3), 'zeros:', np.sum(w_cd == 0))
```

</details>

### Exercise 16 · Logistic regression with L-BFGS
*💻 Code · ★★☆*

Write `logreg_obj(w, X, y, C)` returning the loss **and** gradient of
$\sum_i\log(1+e^{-y_iw^\top x_i}) + \frac{1}{2C}\lVert w\rVert^2$ with labels $y_i\in\{-1,+1\}$, and minimise it with
`scipy.optimize.minimize(..., jac=True, method='L-BFGS-B')`. Compare with `LogisticRegression(C=C, fit_intercept=False)`, which
minimises the same objective scaled by $C$.

In [ ]:
def logreg_obj(w, X, y, C):
    # TODO: return (loss, grad)
    return None

X_lr = rng.normal(size=(300, 4))
y_lr = np.where(X_lr @ np.array([1., -2., 0.5, 0.]) + 0.5 * rng.normal(size=300) > 0, 1, -1)
w_lbfgs = None   # fitted weights

check('L-BFGS weights', w_lbfgs, LogisticRegression(C=1.0, fit_intercept=False, tol=1e-10, max_iter=1000).fit(X_lr, y_lr).coef_.ravel(), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

With $m_i = y_iw^\top x_i$: loss $=\sum$ `np.logaddexp(0, -m)`; gradient $=-X^\top\big(y\,\sigma(-m)\big) + w/C$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

L-BFGS builds a low-rank approximation of the inverse Hessian from the last few gradient differences: near-Newton convergence at $O(md)$ memory. `np.logaddexp` keeps the loss stable for large margins.

```python
from scipy.special import expit
def logreg_obj(w, X, y, C):
    m = y * (X @ w)
    loss = np.logaddexp(0, -m).sum() + w @ w / (2 * C)
    grad = -X.T @ (y * expit(-m)) + w / C
    return loss, grad

X_lr = rng.normal(size=(300, 4))
y_lr = np.where(X_lr @ np.array([1., -2., 0.5, 0.]) + 0.5 * rng.normal(size=300) > 0, 1, -1)
res = minimize(logreg_obj, np.zeros(4), args=(X_lr, y_lr, 1.0), jac=True, method='L-BFGS-B', options={'gtol': 1e-10})
w_lbfgs = res.x
print(res.nit, 'iterations')
```

</details>

### Exercise 17 · Projected gradient descent for a box constraint
*💻 Code · ★★★*

Minimise $f(x) = \lVert Ax-b\rVert^2$ subject to $0\le x_i\le 1$ with projected gradient descent: a GD step followed by clipping
to the box. Use $\eta = 1/(2\lambda_{\max}(A^\top A))$. Compare with `scipy.optimize.minimize(..., bounds=...)`. Verify the KKT
conditions at your solution: for coordinates strictly inside the box the gradient is $\approx 0$; at $x_i=0$ it is $\ge0$; at
$x_i=1$ it is $\le0$.

In [ ]:
A_b = rng.normal(size=(20, 5)); b_b = A_b @ np.array([0.5, 1.5, -0.3, 0.2, 0.9]) + 0.1 * rng.normal(size=20)
x_pgd = None

_r = minimize(lambda x: np.sum((A_b @ x - b_b) ** 2), np.full(5, 0.5), jac=lambda x: 2 * A_b.T @ (A_b @ x - b_b),
              bounds=[(0, 1)] * 5, method='L-BFGS-B', options={'gtol': 1e-12, 'ftol': 1e-15})
check('projected GD solution', x_pgd, _r.x, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

`x = np.clip(x - eta * 2 * A.T @ (A @ x - b), 0, 1)`, repeated a few thousand times.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The unconstrained solution has $x_2\approx1.5>1$ and $x_3<0$; at the optimum $x_2=x_5=1$ and $x_3=0$ are active bounds, and $x_1, x_4$ readjust in the interior (gradient $\approx0$). The KKT sign conditions say the negative gradient points out of the box at active bounds, so no feasible descent direction exists.

```python
A_b = rng.normal(size=(20, 5)); b_b = A_b @ np.array([0.5, 1.5, -0.3, 0.2, 0.9]) + 0.1 * rng.normal(size=20)
eta = 1 / (2 * np.linalg.eigvalsh(A_b.T @ A_b).max())
x = np.full(5, 0.5)
for _ in range(20000):
    x = np.clip(x - eta * 2 * A_b.T @ (A_b @ x - b_b), 0, 1)
x_pgd = x
g = 2 * A_b.T @ (A_b @ x - b_b)
print('x =', np.round(x, 4)); print('grad =', np.round(g, 5))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Calculus and Optimization](Calculus%20and%20Optimization.md).*